# Position-Wise Feed-Forward Network

This section covers the **Position-Wise Feed-Forward Network**. 
Modern language models tend to two main changes:

* they use another activation function
* employ a gating mechanism

We should understand three key areas: 
* **standard feed-forward structures (MLP/FFN)**, 
* **activation functions(ReLU/SiLU)**, and, 
* **Gated Linear Units (GLU)**.

## Feed-Forward Network(FFN) & Multilayer Perceptron(MLP)

In the world of machine learning, **Feedforward Neural Networks (FFN)** and **Multilayer Perceptrons (MLP)** are often used interchangeably. However, while they are closely related, they represent different levels of abstraction.

The simplest way to distinguish them: **All MLPs are FFNs, but not all FFNs are MLPs.**

### 1. Feedforward Neural Network (FFN)

An FFN is a broad category of neural networks where information moves in only one direction: forward. 

There are **no cycles or loops** (FFN <-> RNN).

```
data -> input nodes -> hidden layer -> ... -> hidden layer -> output
```

**Mathematical Representation**

A feedforward network can be viewed as a composition of functions. For a network with  layers, the output  is calculated as:

$$
y = f^{(L)}(f^{(L-1)}(...f^{(1)}(x)...))
$$

Where:
* $x$ is the input vector.
* $f^{(i)}$ represents the transformation at layer $i$.

### 2. Multilayer Perceptron (MLP)


An MLP is a kind of modern of FFN. To be classified as an MLP, a network must meet three criteria:

1. **Multiple Layers:** It must have at least one hidden layer (3 layers total: input, hidden, and output).
2. **Fully Connected (Dense):** Every neuron in layer  must connect to every neuron in layer $i+1$.
3. **Non-linear Activations:** It must use non-linear activation functions (like ReLU or Sigmoid) to avoid collapsing into a simple linear model.

**Mathematical Representation**

For a single layer in an MLP, the output vector  is calculated as:

$$
h = \sigma(W x + b)
$$

Where:
* $x \in \mathbb{R}^n$: Input vector.
* $W \in \mathbb{R}^{m \times n}$: Weight matrix.
* $b \in \mathbb{R}^m$: Bias vector.
* $\sigma$: Non-linear activation function (e.g., $ReLU(z) = \max(0, z)$).

### 3. Direct Comparison


| Feature | Feedforward Neural Network (FFN) | Multilayer Perceptron (MLP) |
| --- | --- | --- |
| **Scope** | A broad class of architectures. | A specific subset of FFNs. |
| **Connectivity** | Can be sparse or locally connected (e.g., CNNs). | **Must** be fully connected (Dense). |
| **Structure** | Unidirectional flow (no loops). | Unidirectional flow with  hidden layer. |
| **Complexity** | Varies from a single layer to billions. | Requires a hidden layer to solve non-linear problems (XOR). |

**Key Distinction: The "XOR" Problem**

Historically, a "Single-Layer Perceptron" (an FFN with no hidden layer) could only solve linearly separable problems. It could not solve the **XOR** logic gate because it couldn't draw a non-linear boundary.

By adding a hidden layer and non-linear activations, it becomes an **MLP**, which gains the power of the **Universal Approximation Theorem**: the ability to approximate any continuous function given enough neurons.


## Activation Functions (ReLU, SiLU)

<img src="../images/SiLU_ReLU.png" width="70%">

### ReLU (Rectified Linear Unit)


$$
ReLU(x) = \max(0, x)
$$

**Characteristics**

* It turns off neurons that have negative values (sets them to 0). 
* ✅ pros
    * The network becomes lighter and more efficient.
* ❌　cons
    * **The "Dying ReLU" Problem:** because of the ReLU turning negative value into zero, the gradient becomes 0, it will stay at 0 forever and that neuron "dies."


### 2. SiLU (Sigmoid Linear Unit)


Also known as **Swish**, SiLU is a more modern, "smooth" version of ReLU. 

$$
SiLU(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}
$$

* Characteristics
* Unlike ReLU, which has a sharp "elbow" at zero, SiLU is smooth everywhere. 

* ✅Pros
    * This helps the optimization process (Gradient Descent) find better minima.
    * Interestingly, for small negative values, SiLU actually dips below zero before returning to zero. This allows some negative information to flow through, which often leads to better accuracy than ReLU.

### 3. Comparison Table


| Feature | ReLU | SiLU (Swish) |
| --- | --- | --- |
| **Formula** | $\max(0, x)$ | $x \cdot \text{sigmoid}(x)$ |
| **Differentiable** | Not at $x=0$ | Yes, everywhere |
| **Computation** | Extremely fast (simple comparison) | Moderate (requires exponential) |
| **Output Range** | $[0, \infty)$ | $[\approx -0.28, \infty)$ |
| **Best For** | General MLPs, CNNs | Deep Transformers, YOLO, LLMs |


## Gated Linear Units (GLUs)

The **Gated Linear Unit (GLU)** is a sophisticated architectural component that moves away from simple "all-or-nothing" activations (like ReLU) toward a **gating mechanism**.

The original definition by Dauphin et al. is:

$$
\text{GLU}(x, W_1, W_2) = \sigma({W_1}x) \odot ({W_2}x)
$$

To visualize what's happening, let's break it into two parallel paths:

1. **The Gate $\sigma({W_1}x)$:** This path applies a sigmoid function, squashing the linear transformation into a range of $[0,1]$. It acts as a learned "filter."
2. **The Content $({W_2}x)$:** This is a standard linear transformation of the input. It carries the actual "data" or features.
3. **The Element-wise Product ($\odot$):** The gate vector multiplies the content vector. If the gate value is $1.0$, the content passes through perfectly; if it's $0.0$, the content is blocked.

---

**🔍Why use GLUs?**

* **Vanishing Gradient Relief:** In a standard network, gradients must pass through non-linearities (like Tanh) at every layer, which can shrink the signal. In a GLU, if the gate is "open" (near 1), the gradient flows through the  path linearly, preserving its strength.
* **Dynamic Selection:** Unlike ReLU, a GLU can choose to block or pass *any* feature based on the context of the input.
* **Reduced Training Bias:** Because they have a linear path, they are easier to train in very deep stacks compared to pure Sigmoid or Tanh networks.

---

**The "SwiGLU" Evolution**

Researchers found that replacing the **Sigmoid** with a **SiLU** (Swish) activation works significantly better.However, we offer no explanation as to why these architectures seem to work; we attribute their success, as all else, to divine benevolence.

The **SwiGLU** variant is defined as:

$$\text{SwiGLU}(x, {W_1}, {W_2}) = \text{SiLU}({W_1}x) \otimes ({W_2}x)$$

In this version, the "gate" isn't just a 0-to-1 filter; it’s a smooth, non-monotonic function that allows the network to learn much more complex representations.


## Code

### Full Code

In [1]:
import torch
from jaxtyping import Float
import torch.nn as nn
from torch.nn.parameter import Parameter
from torch.nn import functional as F

class SwiGLUFFN(nn.Module):

    def __init__(
        self,
        d_model: int,
        d_ff: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    )-> None:
        
        factory_kwargs = {"device": device, "dtype": dtype}
        
        super().__init__()

        self.d_model = d_model
        self.d_ff = d_ff

        self.w1_weight = Parameter(
            torch.empty((self.d_ff, self.d_model), **factory_kwargs)
        )
        self.w2_weight = Parameter(
            torch.empty((self.d_model, self.d_ff), **factory_kwargs)
        )
        self.w3_weight = Parameter(
            torch.empty((self.d_ff, self.d_model), **factory_kwargs)
        )
        self.reset_parameters()
        
    
    def reset_parameters(self) -> None:
        nn.init.trunc_normal_(self.w1_weight)
        nn.init.trunc_normal_(self.w2_weight)
        nn.init.trunc_normal_(self.w3_weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = x @ self.w1_weight.T
        x3 = x @ self.w3_weight.T
        gated = F.silu(x1) * x3
        result = gated @ self.w2_weight.T
        return result



### Step by Step

#### Class initialization

```python
class SwiGLUFFN(nn.Module):

    def __init__(
        self,
        d_model: int,
        d_ff: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    )-> None:
        
        factory_kwargs = {"device": device, "dtype": dtype}
        
        super().__init__()

        self.d_model = d_model
        self.d_ff = d_ff
        # if the glue code provides `d_ff`, you do not need calculation
        # if it doesn't, you have to define d_ff by your self: 
        # d_ff = round(d_model * 8 / 3)

        # Be careful the order of d_ff & d_model, you should leverage row-major
        self.w1_weight = Parameter(
            torch.empty((self.d_ff, self.d_model), **factory_kwargs)
        )
        self.w2_weight = Parameter(
            torch.empty((self.d_model, self.d_ff), **factory_kwargs)
        )
        self.w3_weight = Parameter(
            torch.empty((self.d_ff, self.d_model), **factory_kwargs)
        )
        self.reset_parameters()
```

```python
# You have to initialzie each weight
def reset_parameters(self) -> None:
    nn.init.trunc_normal_(self.w1_weight)
    nn.init.trunc_normal_(self.w2_weight)
    nn.init.trunc_normal_(self.w3_weight)
```

```python
# MUST be ONLY calculation 
def forward(self, x: torch.Tensor) -> torch.Tensor:
    x1 = x @ self.w1_weight.T
    x3 = x @ self.w3_weight.T
    gated = F.silu(x1) * x3
    result = gated @ self.w2_weight.T
    return result

"""
This code was wrong
def forward(self, x: torch.Tensor) -> torch.Tensor: 
    self.w1_weight = Linear(self.d_model, self.d_ff) 
    self.w2_weight = Linear(self.d_ff, self.d_model) 
    self.w3_weight = Linear(self.d_model, self.d_ff) 
    return self.w2_weight(SiLU(self.w1_weight(x)) * self.w3_weight(x))
"""
```

In [5]:
"""
SwiGLUFFN の実装テスト
======================
このスクリプトでは：
  2. Glue code (run_swiglu 関数)
  3. Linearと同様のテストコード
を一つにまとめて実行します。
"""

import torch
import torch.nn as nn
from torch.nn.parameter import Parameter
from torch.nn import functional as F
from torch import Tensor
from jaxtyping import Float


# ============================================================
# SwiGLUFFN の実装
# ============================================================

class SwiGLUFFN(nn.Module):
    """
    SwiGLU Feed-Forward Network

    通常のFFN: output = W2 * ReLU(W1 * x)
    SwiGLU:    output = W2 * (SiLU(W1 * x) ⊗ W3 * x)

    W1 → Swish(SiLU)で活性化してゲートを作る
    W3 → 「通す情報そのもの」を変換する
    両者の要素積がゲーティング機構になる
    W2 → d_ff次元からd_model次元に圧縮して出力
    """

    def __init__(
        self,
        d_model: int,               # 入力・出力の次元数
        d_ff: int,                  # 中間層（拡張後）の次元数
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:

        factory_kwargs = {"device": device, "dtype": dtype}

        super().__init__()

        self.d_model = d_model
        self.d_ff = d_ff

        # W1: (d_ff, d_model) — SiLU に通してゲートを生成する重み
        self.w1_weight = Parameter(
            torch.empty((self.d_ff, self.d_model), **factory_kwargs)
        )
        # W2: (d_model, d_ff) — gated出力をd_modelに圧縮する重み
        self.w2_weight = Parameter(
            torch.empty((self.d_model, self.d_ff), **factory_kwargs)
        )
        # W3: (d_ff, d_model) — ゲートをかける「情報」を生成する重み
        self.w3_weight = Parameter(
            torch.empty((self.d_ff, self.d_model), **factory_kwargs)
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.trunc_normal_(self.w1_weight)
        nn.init.trunc_normal_(self.w2_weight)
        nn.init.trunc_normal_(self.w3_weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., d_model)

        # ゲート: SiLU(W1 * x) — どの情報を通すかを決める
        x1 = x @ self.w1_weight.T          # (..., d_ff)

        # 情報: W3 * x — 何を運ぶか
        x3 = x @ self.w3_weight.T          # (..., d_ff)

        # ゲーティング: 要素積でゲートをかける
        # SiLU(x) = x * sigmoid(x) ≈ Swish(x)
        gated = F.silu(x1) * x3            # (..., d_ff)

        # 出力: W2 でd_model次元に圧縮
        result = gated @ self.w2_weight.T  # (..., d_model)

        return result


# ============================================================
# Glue code
# ============================================================
#
# Glue code とは：
#   実装クラス（SwiGLUFFN）と外部から渡される重みテンソルを
#   「つなぎ合わせる」ための橋渡し関数のこと。
#
#   クラスを直接使うと「インスタンス化 → 重みをセット → forward」
#   という手順を毎回書く必要がある。glue codeはこれを一関数に
#   まとめることで、テストや評価スクリプトから簡潔に呼べるようにする。
#
#   重みを外から渡す設計にすることで、
#   「学習済みモデルの重みをロードして推論する」
#   「テストごとに異なる重みで挙動を確認する」
#   などのユースケースに対応しやすくなる。
# ============================================================

def run_swiglu(
    d_model: int,
    d_ff: int,
    w1_weight: Float[Tensor, " d_ff d_model"],
    w2_weight: Float[Tensor, " d_model d_ff"],
    w3_weight: Float[Tensor, " d_ff d_model"],
    in_features: Float[Tensor, " ... d_model"],
) -> Float[Tensor, " ... d_model"]:
    """
    外部から重みを受け取って SwiGLUFFN を実行する関数。

    Args:
        d_model: 入力・出力の次元数
        d_ff:    中間層の次元数
        w1_weight: ゲート用の重み  shape=(d_ff, d_model)
        w2_weight: 出力圧縮用の重み shape=(d_model, d_ff)
        w3_weight: 情報運搬用の重み shape=(d_ff, d_model)
        in_features: 入力テンソル shape=(..., d_model)
                     '...' は任意の先行次元（batch, seq_len など）

    Returns:
        入力と同じshapeの出力テンソル shape=(..., d_model)
    """

    # SwiGLUFFN のインスタンスを作る（重みはランダム初期化される）
    swiglu = SwiGLUFFN(d_model, d_ff)

    # 外部から渡された重みをセットする
    # .data を使う理由:
    #   直接 `swiglu.w1_weight = w1_weight` と書くと、
    #   PyTorch は w1_weight を Parameter ではなく
    #   ただの Tensor として扱ってしまい、
    #   勾配追跡の対象から外れてしまう。
    #   `.data` に代入することで Parameter のメタ情報を保ちながら
    #   中身の値だけを差し替えることができる。
    swiglu.w1_weight.data = w1_weight
    swiglu.w2_weight.data = w2_weight
    swiglu.w3_weight.data = w3_weight

    return swiglu(in_features)


# ============================================================
# テストコード（Linearのテストと同じ形式）
# ============================================================

def main():
    print("=" * 60)
    print("SwiGLUFFN テスト")
    print("=" * 60)

    # 1. 次元の設定
    batch_size = 4
    seq_len    = 6       # 系列長（トークン数の例）
    d_model    = 8       # 入力・出力の次元
    d_ff       = 32      # 中間層の次元（d_model × 4 が典型的）

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")

    # --------------------------------------------------------
    # テスト A: SwiGLUFFN クラスを直接使う
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト A: SwiGLUFFN クラスの直接使用")
    print("-" * 40)

    model = SwiGLUFFN(d_model, d_ff, device=device)

    # 2D入力: (batch_size, d_model)
    input_2d: Float[Tensor, "batch d_model"] = torch.randn(batch_size, d_model).to(device)
    output_2d = model(input_2d)

    print(f"\n[2D入力] shape: {input_2d.shape}")
    print(f"[2D出力] shape: {output_2d.shape}  ← d_model={d_model} に戻っている")

    # 3D入力: (batch_size, seq_len, d_model) — Transformerでよく使う形
    input_3d: Float[Tensor, "batch seq d_model"] = torch.randn(batch_size, seq_len, d_model).to(device)
    output_3d = model(input_3d)

    print(f"\n[3D入力] shape: {input_3d.shape}  ← batch × seq × d_model")
    print(f"[3D出力] shape: {output_3d.shape}  ← shapeが保たれている ('...'の意味)")

    # 重みの形状確認
    print(f"\nW1 shape: {model.w1_weight.shape}  (d_ff={d_ff}, d_model={d_model})")
    print(f"W2 shape: {model.w2_weight.shape}  (d_model={d_model}, d_ff={d_ff})")
    print(f"W3 shape: {model.w3_weight.shape}  (d_ff={d_ff}, d_model={d_model})")

    # --------------------------------------------------------
    # テスト B: Glue code (run_swiglu) を使う
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト B: run_swiglu (glue code) 経由")
    print("-" * 40)

    # 重みを外から用意して渡す（学習済みモデルのロードを模した状況）
    w1 = torch.randn(d_ff, d_model)
    w2 = torch.randn(d_model, d_ff)
    w3 = torch.randn(d_ff, d_model)

    input_tensor = torch.randn(batch_size, d_model)
    output_glue  = run_swiglu(d_model, d_ff, w1, w2, w3, input_tensor)

    print(f"\n入力  shape: {input_tensor.shape}")
    print(f"出力  shape: {output_glue.shape}  ← d_model={d_model} に戻っている")

    # --------------------------------------------------------
    # テスト C: 同じ重みで直接クラスと glue code が一致するか確認
    # --------------------------------------------------------
    print("\n" + "-" * 40)
    print("テスト C: クラス直接 vs glue code — 出力が一致するか")
    print("-" * 40)

    w1 = torch.randn(d_ff, d_model)
    w2 = torch.randn(d_model, d_ff)
    w3 = torch.randn(d_ff, d_model)
    x  = torch.randn(batch_size, d_model)

    # クラス直接
    model_c = SwiGLUFFN(d_model, d_ff)
    model_c.w1_weight.data = w1
    model_c.w2_weight.data = w2
    model_c.w3_weight.data = w3
    out_direct = model_c(x)

    # glue code 経由
    out_glue = run_swiglu(d_model, d_ff, w1, w2, w3, x)

    match = torch.allclose(out_direct, out_glue, atol=1e-6)
    print(f"\n出力一致: {match}  ({'OK' if match else 'NG — 実装を確認してください'})")
    print(f"最大差分: {(out_direct - out_glue).abs().max().item():.2e}")

    print("\n" + "=" * 60)
    print("すべてのテスト完了")
    print("=" * 60)


if __name__ == "__main__":
    main()

SwiGLUFFN テスト

Device: cpu

----------------------------------------
テスト A: SwiGLUFFN クラスの直接使用
----------------------------------------

[2D入力] shape: torch.Size([4, 8])
[2D出力] shape: torch.Size([4, 8])  ← d_model=8 に戻っている

[3D入力] shape: torch.Size([4, 6, 8])  ← batch × seq × d_model
[3D出力] shape: torch.Size([4, 6, 8])  ← shapeが保たれている ('...'の意味)

W1 shape: torch.Size([32, 8])  (d_ff=32, d_model=8)
W2 shape: torch.Size([8, 32])  (d_model=8, d_ff=32)
W3 shape: torch.Size([32, 8])  (d_ff=32, d_model=8)

----------------------------------------
テスト B: run_swiglu (glue code) 経由
----------------------------------------

入力  shape: torch.Size([4, 8])
出力  shape: torch.Size([4, 8])  ← d_model=8 に戻っている

----------------------------------------
テスト C: クラス直接 vs glue code — 出力が一致するか
----------------------------------------

出力一致: True  (OK)
最大差分: 0.00e+00

すべてのテスト完了
